# Ensemble Methods and Uncertainty Quantification

This notebook demonstrates how to combine multiple models into a powerful ensemble
and how to produce well-calibrated uncertainty estimates using Endgame:

1. Train individual base models (LightGBM, XGBoost, Random Forest)
2. Combine them with a **Super Learner** ensemble (cross-validated optimal blending)
3. Wrap the best model with **Conformal Prediction** for coverage-guaranteed prediction sets
4. Apply **Venn-ABERS** calibration for well-calibrated probability intervals

In [ ]:
import endgame as eg
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

## 1. Load Data

We use the Breast Cancer Wisconsin dataset -- a binary classification task with
30 numeric features and 569 samples. We split into train, calibration, and test sets
since conformal prediction and Venn-ABERS require a held-out calibration set.

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset: {data.DESCR.splitlines()[0]}")
print(f"Shape: {X.shape}")
print(f"Classes: {dict(zip(data.target_names, np.bincount(y)))}")

# First split: 70% train, 30% temp (for calibration + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 50/50 calibration and test from the temp set
X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTrain: {X_train.shape[0]}, Calibration: {X_cal.shape[0]}, Test: {X_test.shape[0]}")

## 2. Train Individual Base Models

We train three diverse base models:
- **LGBMWrapper**: LightGBM with Endgame's competition-tuned presets
- **XGBWrapper**: XGBoost with competition-tuned presets
- **RandomForestClassifier**: Sklearn's random forest as a complementary learner

Diversity among base learners is critical for ensemble performance.

In [ ]:
from endgame.models import LGBMWrapper, XGBWrapper

# Model 1: LightGBM
lgbm = LGBMWrapper(preset="endgame", verbose=-1)
lgbm.fit(X_train, y_train, eval_set=[(X_test, y_test)])

# Model 2: XGBoost
xgb = XGBWrapper(preset="endgame", verbosity=0)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)])

# Model 3: Random Forest
rf = RandomForestClassifier(n_estimators=500, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate each model
print("Individual Model Performance")
print("=" * 50)
for name, model in [("LightGBM", lgbm), ("XGBoost", xgb), ("RandomForest", rf)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"  {name:15s}  Accuracy: {acc:.4f}  AUC: {auc:.4f}")

## 3. Super Learner Ensemble

The **Super Learner** (van der Laan et al., 2007) is an oracle-optimal ensemble method.
It uses cross-validation to generate out-of-fold predictions from each base learner,
then solves for the optimal convex combination using non-negative least squares (NNLS).

Key properties:
- Asymptotically at least as good as the best single model
- NNLS constraint ensures non-negative, normalized weights
- Adding a weak learner never hurts (weight goes to zero)

In [ ]:
from endgame.ensemble import SuperLearner

sl = SuperLearner(
    base_estimators=[
        ("lgbm", LGBMWrapper(preset="endgame", verbose=-1)),
        ("xgb", XGBWrapper(preset="endgame", verbosity=0)),
        ("rf", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
        ("lr", LogisticRegression(max_iter=1000, random_state=42)),
    ],
    meta_learner="nnls",
    cv=5,
    random_state=42,
    verbose=True,
)

sl.fit(X_train, y_train)

# Show ensemble weights
print("\nSuper Learner Weights")
print("=" * 50)
for (name, _), weight in zip(sl.base_estimators, sl.coef_):
    bar = "#" * int(weight * 40)
    print(f"  {name:15s}  {weight:.4f}  {bar}")

# Show CV scores for each base learner
print("\nCross-Validated Scores (R-squared on OOF predictions)")
print("=" * 50)
for name, score in sl.cv_scores_.items():
    print(f"  {name:15s}  {score:.4f}")

In [ ]:
# Compare Super Learner vs individual models on the test set
y_pred_sl = sl.predict(X_test)
y_proba_sl = sl.predict_proba(X_test)[:, 1]

print("Test Set Comparison")
print("=" * 55)
for name, model in [("LightGBM", lgbm), ("XGBoost", xgb), ("RandomForest", rf)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"  {name:15s}  Accuracy: {acc:.4f}  AUC: {auc:.4f}")

acc_sl = accuracy_score(y_test, y_pred_sl)
auc_sl = roc_auc_score(y_test, y_proba_sl)
print(f"  {'SuperLearner':15s}  Accuracy: {acc_sl:.4f}  AUC: {auc_sl:.4f}  <--")

## 4. Conformal Prediction

**Conformal prediction** wraps any classifier to produce *prediction sets* with a
formal coverage guarantee: P(y_true in prediction_set) >= 1 - alpha.

For example, with `alpha=0.1`, at least 90% of test samples will have the true label
included in their prediction set. This guarantee holds under the exchangeability
assumption (i.i.d. data satisfies this).

We use the **Adaptive Prediction Sets (APS)** method, which produces randomized
sets with exact finite-sample coverage.

In [ ]:
from endgame.calibration import ConformalClassifier

# Wrap the LightGBM model with conformal prediction
# alpha=0.1 means we target 90% coverage
cp = ConformalClassifier(
    estimator=LGBMWrapper(preset="endgame", verbose=-1),
    method="aps",
    alpha=0.1,
    random_state=42,
)

# Fit on training data, calibrate on calibration data
cp.fit(X_train, y_train, X_cal=X_cal, y_cal=y_cal)

# Get prediction sets on the test data
prediction_sets = cp.predict(X_test)

# Compute coverage and set sizes
coverage = cp.coverage_score(X_test, y_test)
avg_size = cp.average_set_size(X_test)

print("Conformal Prediction Results (alpha=0.1, target coverage=90%)")
print("=" * 55)
print(f"  Empirical coverage:      {coverage:.3f}")
print(f"  Average set size:        {avg_size:.3f}")
print(f"  Singleton sets (certain): {sum(len(s) == 1 for s in prediction_sets)}")
print(f"  Two-class sets (unsure):  {sum(len(s) == 2 for s in prediction_sets)}")
print(f"  Empty sets:               {sum(len(s) == 0 for s in prediction_sets)}")

In [ ]:
# Show some example prediction sets
class_names = data.target_names  # ['malignant', 'benign']

print("Example Prediction Sets (first 15 test samples)")
print("=" * 65)
print(f"  {'Sample':>6s}  {'True Label':>12s}  {'Prediction Set':>30s}  {'Covered?':>8s}")
print("-" * 65)
for i in range(min(15, len(y_test))):
    true_label = class_names[y_test[i]]
    pred_set_names = {class_names[c] for c in prediction_sets[i]}
    covered = y_test[i] in prediction_sets[i]
    print(f"  {i:6d}  {true_label:>12s}  {str(pred_set_names):>30s}  {'Yes' if covered else 'No':>8s}")

## 5. Venn-ABERS Calibration

**Venn-ABERS** provides probability intervals [p0, p1] rather than point estimates.
For each test point, it computes a lower (p0) and upper (p1) probability bound
using isotonic regression on an augmented calibration set.

Key advantages over standard calibration methods:
- Theoretical validity guarantees under exchangeability
- Distribution-free (no parametric assumptions)
- Provides uncertainty about the probability estimate itself

We compare uncalibrated vs Venn-ABERS calibrated probabilities using the Brier score
(lower is better).

In [ ]:
from endgame.calibration import VennABERS

# Wrap a LightGBM model with Venn-ABERS calibration
va = VennABERS(
    estimator=LGBMWrapper(preset="endgame", verbose=-1),
)

# Fit on training data, calibrate on calibration data
va.fit(X_train, y_train, X_cal=X_cal, y_cal=y_cal)

# Get calibrated probabilities
va_proba = va.predict_proba(X_test)[:, 1]

# Get probability intervals
p0, p1 = va.predict_proba_interval(X_test)

# Compare with uncalibrated probabilities from the raw model
uncal_proba = lgbm.predict_proba(X_test)[:, 1]

brier_uncal = brier_score_loss(y_test, uncal_proba)
brier_va = brier_score_loss(y_test, va_proba)

print("Calibration Comparison")
print("=" * 50)
print(f"  Uncalibrated Brier Score:  {brier_uncal:.4f}")
print(f"  Venn-ABERS Brier Score:    {brier_va:.4f}")
print(f"  Improvement:               {brier_uncal - brier_va:+.4f}")

In [ ]:
# Show probability intervals for some test samples
# The interval [p0, p1] quantifies uncertainty about the probability itself
interval_widths = va.interval_width(X_test)

print("Venn-ABERS Probability Intervals (first 15 test samples)")
print("=" * 75)
print(f"  {'Sample':>6s}  {'True':>5s}  {'p0 (lower)':>10s}  {'p1 (upper)':>10s}  {'Width':>7s}  {'VA Prob':>8s}  {'Raw Prob':>8s}")
print("-" * 75)
for i in range(min(15, len(y_test))):
    print(
        f"  {i:6d}  {y_test[i]:5d}  {p0[i]:10.4f}  {p1[i]:10.4f}  "
        f"{interval_widths[i]:7.4f}  {va_proba[i]:8.4f}  {uncal_proba[i]:8.4f}"
    )

print(f"\nAverage interval width: {np.mean(interval_widths):.4f}")
print(f"Median interval width:  {np.median(interval_widths):.4f}")

## Summary

In this notebook we demonstrated three complementary techniques for building
robust and trustworthy classifiers:

1. **Super Learner Ensemble**: Combines multiple diverse base learners using
   cross-validated NNLS weights. The ensemble is guaranteed to be asymptotically
   at least as good as the best individual model, and in practice often improves
   accuracy by leveraging complementary model strengths.

2. **Conformal Prediction**: Wraps any classifier to produce prediction sets with
   a formal coverage guarantee (e.g., 90% of test samples will contain the true
   label). When the model is uncertain, it outputs both classes rather than making
   a potentially incorrect point prediction. This is especially valuable in
   safety-critical applications like medical diagnosis.

3. **Venn-ABERS Calibration**: Produces probability intervals [p0, p1] instead of
   a single probability estimate. These intervals have theoretical calibration
   guarantees and quantify the uncertainty in the probability estimate itself.
   Compared to uncalibrated probabilities, Venn-ABERS typically achieves lower
   Brier scores.

**Key takeaways:**
- Ensembles improve accuracy by combining diverse learners
- Conformal prediction gives distribution-free coverage guarantees
- Venn-ABERS gives well-calibrated probability estimates with validity guarantees
- All three methods are model-agnostic and work with any sklearn-compatible estimator

Next steps:
- Try `ConformalRegressor` for prediction intervals in regression tasks
- Explore `HillClimbingEnsemble` for greedy forward-selection ensembles
- Use `StackingEnsemble` for meta-learner stacking with arbitrary second-level models